# Drone Eyes & Ears -- NC-SSM/NC-TCN + Mamba-COP-RFS Demo

**Concept**: a single laptop notebook combines the full NC family on a simulated drone payload.

| Role | Model | Footprint | Latency |
|---|---|---|---|
| ears | NC-SSM (KWS) | 7.4 KB INT8 | < 1 ms |
| ears | NC-TCN (KWS) | 21 KB INT8 | < 1 ms |
| eyes | Mamba-COP-RFS (DOA + GM-PHD) | 41.4 KB INT8 | 3.4 ms / scan |

**Scenario**: Search-and-Rescue. Drone hovers over a debris field; 4 voices call out ("help", "water", "down", "here"), one is briefly occluded then re-acquired thanks to the SSM hidden state.

**Pipeline per scan**:
1. 8-mic ULA acoustic snapshot (synthetic).
2. 4th-order COP DOA estimation (181-angle pseudo-spectrum).
3. Physics-based GM-PHD tracker (predict -> identify -> update).
4. Per-track KWS classification (NC-SSM/NC-TCN -- stub or real).

**Output**: 4-panel summary figure (live spectrum heatmap, polar DOA, GM-PHD tracks, KWS timeline).

## 1. Setup

In [ ]:
import sys
from pathlib import Path

REPO = Path('.').resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# Reuse all the heavy lifting from drone_demo.py -- one source of truth.
from drone_demo import (SCENARIOS, KW_COLOURS,
                        build_trajectory, run_pipeline,
                        attach_kws_labels, render)

print('Available scenarios:', list(SCENARIOS))
print('Keyword palette   :', list(KW_COLOURS))

## 2. Pick a scenario

Two pre-built scenarios; both use 8-mic ULA, 30 scans, mid-SNR (10-12 dB), and at least one occlusion event.

In [ ]:
scenario = SCENARIOS['sar']     # change to 'combat' for the military variant
print(f"=== {scenario['name']} ===")
print(f"  scans     : {scenario['n_scans']}")
print(f"  sources   : {len(scenario['sources'])}")
print(f"  occlusions: {scenario['occlusions']}")
print(f"  SNR (dB)  : {scenario['snr_db']}")

## 3. Run the pipeline (COP-DOA + GM-PHD)

On a laptop CPU this takes ~5 seconds for the full 30-scan scenario.

In [ ]:
import time
t0 = time.time()
history = run_pipeline(scenario)
labelled = attach_kws_labels(history, scenario)
print(f'pipeline elapsed: {time.time()-t0:.2f} s')
print(f'spectra shape  : {np.stack(history["spectra"]).shape}')
print(f'last-scan tracks: {labelled[-1]}')

## 4. Inspect a single scan

Quickly check the COP pseudo-spectrum at the middle of the scenario; the four bright peaks correspond to the four sources (one is masked by the occlusion window depending on the scan).

In [ ]:
scan_idx = 15
angles = history['scan_angles_deg']
spec = history['spectra'][scan_idx]

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(angles, 20*np.log10(spec/spec.max() + 1e-3), color='#2C3E50')
for tid, deg, kw in labelled[scan_idx]:
    col = KW_COLOURS.get(kw, '#888')
    ax.axvline(deg, color=col, lw=2, alpha=0.6)
    ax.text(deg, 0.5, f'{kw} (id {tid})', color=col,
            fontsize=10, fontweight='bold', ha='center')
ax.set_xlabel('DOA (degrees)')
ax.set_ylabel('COP pseudo-spectrum (dB, normalised)')
ax.set_title(f'Scan {scan_idx}: {len(labelled[scan_idx])} tracks detected')
ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 5. Full 4-panel summary

(a) COP heatmap over time | (b) polar DOA + KWS @ mid-scan | (c) GM-PHD tracks with KWS captions | (d) KWS timeline per track.

In [ ]:
render(history, labelled, scenario, REPO / 'drone_demo.png')
from IPython.display import Image
Image(str(REPO / 'drone_demo.png'))

## 6. What's running where (edge deployment summary)

| Stage | Cost | Where it runs |
|---|---|---|
| COP-4th DOA | $O(M^4 T)$ -- 3.17 ms | STM32H7 Cortex-M7 |
| SSM hidden-state step | $O(d_s)$ -- 0.08 ms | STM32H7 |
| Actor-critic MLP | $O(d_h^2)$ -- 0.06 ms | STM32H7 |
| GM-PHD update | $O(J_t)$ -- 0.09 ms | STM32H7 |
| **Total per scan** | **3.4 ms** | **single MCU** |

Total model footprint: **41.4 KB INT8** for the RL policy + **7.4 KB** for NC-SSM KWS = under **50 KB** for the full eyes-and-ears stack.

## 7. Next steps

- `drone_demo_streamlit.py` -- interactive dashboard for live pitching.
- `drone_demo_anim.py` -- 30 fps MP4 / GIF for video pitches.
- `drone_demo_mic.py` -- live capture from a USB mic array (ReSpeaker 6-mic).
- `drone_demo.py --real-kws` -- drop the keyword stub for actual NC-SSM/NC-TCN inference.

Code: <https://github.com/DrJinHoChoi/IronDome-DOA-Tracking>  
License: dual academic + commercial (see `LICENSE`).  
Contact: jinhochoi@smartear.co.kr